# 11주 1차시 · 분류의 개념: 의사결정나무와 과적합

**학습 목표**
- 분류(범주 판정)와 회귀(숫자 예측)의 차이를 코드로 확인한다.
- 의사결정나무를 학습시키고, 그림(`plot_tree`)과 규칙(`export_text`)으로 읽는다.
- 나무가 질문을 고르는 기준(섞임 정도 = 지니 불순도)을 직접 계산해 본다.
- 나무를 깊게 키우면 훈련 정확도와 시험 정확도가 어떻게 달라지는지(과적합) 확인한다.

**사용 데이터** `process.csv` — 사출 성형 공정 기록 494개(가상 데이터).
입력은 공정 조건 5개(`temp`, `pressure`, `speed`, `humidity`, `thickness`), 정답은 `defect`(0 = 정상, 1 = 불량)입니다.

맨 위 셀부터 **순서대로** 실행하세요.

## 1. 준비: 한글 그래프 설정과 라이브러리 불러오기

In [ ]:
!pip -q install koreanize-matplotlib
import koreanize_matplotlib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

## 2. 데이터 불러오기
불량(`defect` = 1)이 몇 개인지, 비율은 얼마인지 먼저 봅니다.

In [ ]:
DATA = 'https://raw.githubusercontent.com/Daeil-Noh/Lectures/main/ax-microdegree/ai-data-analysis/data/'
df = pd.read_csv(DATA + 'process.csv')
print(df.shape)                          # (행 수, 열 수)
print(df['defect'].value_counts())       # 0 = 정상, 1 = 불량
print('불량률:', round(df['defect'].mean() * 100, 1), '%')

## 3. 입력(X)과 정답(y) 정하고, 훈련용·시험용으로 나누기
- `strength`(인장강도)는 제품을 당겨 끊어 봐야 아는 값이라 **생산 직후 판정**에는 쓸 수 없으므로 입력에서 뺍니다.
- 9주 2차시처럼 `train_test_split` 으로 나누되, 이번에는 **훈련 70% · 시험 30%** 입니다. 20%로 나누면 시험 데이터의 불량이 14개뿐이라 “불량을 잘 잡는지” 재기에 너무 적기 때문입니다.
- `stratify=y` 는 두 쪽의 불량 비율을 같게 맞추는 옵션입니다(9주 2차시에서 배운 층화).

In [ ]:
features = ['temp', 'pressure', 'speed', 'humidity', 'thickness']
X = df[features]
y = df['defect']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('훈련:', len(X_train), '개 / 시험:', len(X_test), '개')

## 4. 작은 나무(최대 깊이 2) 학습하기
`max_depth=2` → 질문을 최대 두 번까지 연달아 합니다. `fit` 하는 순간 컴퓨터가 질문을 고릅니다.

In [ ]:
tree2 = DecisionTreeClassifier(max_depth=2, random_state=42)
tree2.fit(X_train, y_train)

print('훈련 정확도:', round(tree2.score(X_train, y_train), 3))
print('시험 정확도:', round(tree2.score(X_test, y_test), 3))

## 5. 나무 그림 보기
상자마다 위에서부터 **질문**, gini(섞임 정도), samples(도착한 훈련 데이터 제품 수), value([정상 수, 불량 수]), class(판정)가 적혀 있습니다.
질문이 참이면 **왼쪽**, 거짓이면 **오른쪽**으로 내려갑니다.

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(tree2, feature_names=features, class_names=['정상', '불량'],
          filled=True, rounded=True, fontsize=11)
plt.title('의사결정나무 (max_depth=2)')
plt.show()

## 6. 나무를 글자 규칙으로 보기
`export_text` 는 같은 나무를 `if` 규칙처럼 보여 줍니다. 슬라이드의 나무와 숫자를 비교해 보세요.

In [ ]:
print(export_text(tree2, feature_names=features))

# 손으로 만든 제품 하나를 판정해 보기 (값을 바꿔 보세요)
new = pd.DataFrame([[212, 80, 1200, 79.0, 2.01]], columns=features)
print('판정:', tree2.predict(new)[0], '(0 = 정상, 1 = 불량)')

## 7. 나무는 첫 질문을 어떻게 골랐을까? — 섞임 정도(지니 불순도) 직접 계산
섞임 정도 = 1 − (정상 비율² + 불량 비율²). 0이면 한 종류만 있고, 0.5면 반반입니다.
질문으로 나눈 두 그룹의 섞임 정도를 **제품 수로 가중 평균**해서, 가장 낮아지는 질문이 좋은 질문입니다.

In [ ]:
def gini(labels):
    p = labels.mean()                    # 불량 비율
    return 1 - (p ** 2 + (1 - p) ** 2)

def split_score(col, threshold):
    left = y_train[X_train[col] <= threshold]     # 질문에 '예'인 그룹
    right = y_train[X_train[col] > threshold]     # '아니오' 그룹
    n = len(y_train)
    return (len(left) * gini(left) + len(right) * gini(right)) / n

print('나누기 전   :', round(gini(y_train), 3))
print('질문 A 압력 ≤ 79.15 :', round(split_score('pressure', 79.15), 3))
print('질문 B 습도 ≤ 76.2  :', round(split_score('humidity', 76.2), 3))

## 8. 나무를 점점 깊게 키우며 훈련·시험 정확도 비교하기
`max_depth` 를 1부터 11까지 바꿔 가며 같은 데이터로 학습하고, 결과를 표로 모읍니다.

In [ ]:
rows = []
for d in range(1, 12):
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    rows.append({'max_depth': d,
                 '잎 수': m.get_n_leaves(),
                 '훈련 정확도': round(m.score(X_train, y_train), 3),
                 '시험 정확도': round(m.score(X_test, y_test), 3)})
result = pd.DataFrame(rows)
print(result)

## 9. 그래프로 과적합 확인하기
훈련 정확도는 계속 오르는데 시험 정확도는 어디서 가장 높고, 그 뒤에 어떻게 되나요?
점선은 모든 제품을 “정상”이라고만 했을 때의 시험 정확도입니다.

> **용어 주의** 여러 `max_depth` 중 하나를 **고르는 데** 쓴 데이터는 원래 **검증 데이터(validation)** 의 역할입니다. 실무에서는 훈련 데이터에서 검증 데이터를 한 번 더 떼어 설정을 고르고, **시험 데이터는 다 고른 뒤 마지막에 한 번만** 채점합니다. 이 실습은 데이터가 작아 시험 데이터로 비교하므로, 여기서 고른 깊이의 시험 정확도는 실제보다 조금 좋게 나올 수 있습니다.

In [ ]:
baseline = (y_test == 0).mean()
plt.figure(figsize=(8, 4))
plt.plot(result['max_depth'], result['훈련 정확도'], marker='o', label='훈련 정확도')
plt.plot(result['max_depth'], result['시험 정확도'], marker='o', label='시험 정확도')
plt.axhline(baseline, color='gray', linestyle='--', label='전부 정상이라고 할 때')
plt.xlabel('최대 깊이 (max_depth)')
plt.ylabel('정확도')
plt.title('나무가 깊어질수록: 훈련 vs 시험')
plt.legend()
plt.show()

## 10. 깊이 제한이 없으면?
`max_depth` 를 주지 않으면 나무는 훈련 데이터 제품을 전부 맞힐 때까지 자랍니다.

In [ ]:
full = DecisionTreeClassifier(random_state=42)
full.fit(X_train, y_train)
print('깊이:', full.get_depth(), '/ 잎 수:', full.get_n_leaves())
print('훈련 정확도:', round(full.score(X_train, y_train), 3))
print('시험 정확도:', round(full.score(X_test, y_test), 3))

## 직접 바꿔 보기
1. `MY_DEPTH` 를 1, 3, 5, 8로 바꿔 실행해 보세요. 나무 그림과 정확도가 어떻게 바뀌나요?
2. `MY_LEAF` (잎 하나에 있어야 할 최소 훈련 데이터 제품 수)를 1, 5, 10, 20으로 바꿔 보세요. 과적합이 줄어드나요?
3. 7번 셀에서 질문의 특성과 기준값(예: `'temp', 223.35` 또는 `'thickness', 1.95`)을 바꿔 섞임 정도를 계산해 보세요.

In [ ]:
MY_DEPTH = 3        # ← 바꿔 보기
MY_LEAF = 1         # ← 바꿔 보기 (min_samples_leaf)

my_tree = DecisionTreeClassifier(max_depth=MY_DEPTH, min_samples_leaf=MY_LEAF, random_state=42)
my_tree.fit(X_train, y_train)
print('잎 수:', my_tree.get_n_leaves())
print('훈련 정확도:', round(my_tree.score(X_train, y_train), 3))
print('시험 정확도:', round(my_tree.score(X_test, y_test), 3))

plt.figure(figsize=(16, 6))
plot_tree(my_tree, feature_names=features, class_names=['정상', '불량'], filled=True, fontsize=8)
plt.show()

In [ ]:
# 3번 과제: 특성 이름과 기준값을 바꿔 보세요
print('내 질문의 섞임 정도:', round(split_score('temp', 223.35), 3))

## 정리
- **분류**는 정상/불량처럼 정해진 범주 중 하나를 고르는 지도학습입니다.
- **의사결정나무**는 섞임 정도(지니 불순도)를 가장 많이 줄이는 예/아니오 질문을 반복해서 만든 규칙입니다.
- 나무를 깊게 키우면 훈련 정확도는 100%까지 오르지만 시험 정확도는 오히려 떨어질 수 있습니다 — **과적합**. 모델의 실력은 시험 데이터로 잽니다.

**생성형AI에게 물어볼 만한 질문**
- “sklearn plot_tree 그림에서 gini, samples, value가 각각 무슨 뜻인지 고등학생도 알게 설명해 줘.”
- “의사결정나무에서 max_depth와 min_samples_leaf를 바꾸면 과적합이 어떻게 달라지는지, 이 코드로 확인하는 방법을 알려 줘.” — 답에 나온 코드는 직접 실행해 결과를 확인하세요.